# Data Cleaning & Preparation Project

## Objective
The objective of this project is to clean and prepare a raw e-commerce dataset by:
- Handling missing values
- Removing duplicate records
- Correcting inconsistent formats
- Validating data integrity

## Tools Used
- Python
- Pandas
- Jupyter Notebook

## Dataset Description
The dataset contains e-commerce transaction records including:
- Orders
- Customers
- Products
- Payments
- Shipping details

## Key Tasks
1. Data Inspection
2. Missing Value Handling
3. Duplicate Removal
4. Data Type Correction
5. Standardization
6. Validation Checks
7. Export Cleaned Dataset

In [2]:
import pandas as pd
import numpy as np


In [3]:
import sys
print(sys.executable)

/workspaces/Decode_Labs_Tech_Internship/.venv/bin/python


In [4]:
df = pd.read_excel("/workspaces/Decode_Labs_Tech_Internship/data/Dataset for Data Analytics.xlsx")
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [5]:
df.shape
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   str           
 1   Date             1200 non-null   datetime64[us]
 2   CustomerID       1200 non-null   str           
 3   Product          1200 non-null   str           
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   str           
 7   PaymentMethod    1200 non-null   str           
 8   OrderStatus      1200 non-null   str           
 9   TrackingNumber   1200 non-null   str           
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    str           
 12  ReferralSource   1200 non-null   str           
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(9)


OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

# Missing Value Analysis

The dataset was checked for missing values to identify incomplete records.

Observation:
- Most columns contain complete data.
- The `CouponCode` column contains missing values.

Interpretation:
The missing values in `CouponCode` are expected because not all customers use discount coupons during purchases.

Decision:
The missing values in `CouponCode` will be retained and labeled as `"No Coupon"` for better readability and consistency during analysis.

In [6]:
df["CouponCode"] = df["CouponCode"].fillna("No Coupon")
df.isnull().sum()   

OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64

# Duplicate Record Analysis

The dataset was checked for:
- Fully duplicated rows
- Duplicate Order IDs
- Duplicate Tracking Numbers

Duplicate records can inflate business metrics and create inaccurate analysis results.

The objective is to ensure each transaction is uniquely represented.

In [7]:
df.duplicated().sum()
df["OrderID"].duplicated().sum()
df["TrackingNumber"].duplicated().sum()

np.int64(0)

# Data Type & Format Validation

The dataset was validated to ensure:
- Date fields are properly formatted
- Numerical values are logically valid
- Text fields are standardized for consistency

This step improves reliability for future analysis and visualization tasks.

In [8]:
df[df["Quantity"] <= 0]
df[df["UnitPrice"] <= 0]
df[df["TotalPrice"] <= 0]

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice


In [9]:
df["CalculatedPrice"] = df["Quantity"] * df["UnitPrice"]

invalid_total = df[df["CalculatedPrice"] != df["TotalPrice"]]

invalid_total.shape

(107, 15)

In [10]:
invalid_total[
    ["Quantity", "UnitPrice", "TotalPrice", "CalculatedPrice"]
].head(10)

,Quantity,UnitPrice,TotalPrice,CalculatedPrice
2,5,550.68,2753.40,2753.40
10,5,625.97,3129.85,3129.85
11,3,49.14,147.42,147.42
32,5,536.72,2683.60,2683.60
41,3,611.45,1834.35,1834.35
43,3,499.21,1497.63,1497.63
55,3,344.40,1033.20,1033.20
56,3,479.21,1437.63,1437.63
64,3,130.61,391.83,391.83
69,3,478.65,1435.95,1435.95


# Financial Validation Check

A validation check was performed to verify that:

TotalPrice = Quantity × UnitPrice

Initial validation identified mismatches caused by floating-point precision behavior in Python.

To ensure accurate comparison for financial values, both calculated and stored prices were rounded to two decimal places before validation.

In [11]:
df["CalculatedPrice"] = (
    df["Quantity"] * df["UnitPrice"]
).round(2)

invalid_total = df[
    df["CalculatedPrice"] != df["TotalPrice"]
]

invalid_total.shape

(29, 15)

In [12]:
invalid_total["Difference"] = (
    invalid_total["CalculatedPrice"] - invalid_total["TotalPrice"]
)

invalid_total[
    [
        "Quantity",
        "UnitPrice",
        "TotalPrice",
        "CalculatedPrice",
        "Difference"
    ]
].head(15)

,Quantity,UnitPrice,TotalPrice,CalculatedPrice,Difference
39,3,256.46,769.38,769.38,1.136868e-13
66,5,127.18,635.90,635.90,-1.136868e-13
85,3,146.73,440.19,440.19,1.136868e-13
111,3,309.22,927.66,927.66,-1.136868e-13
113,5,190.08,950.40,950.40,-1.136868e-13
123,3,278.04,834.12,834.12,-1.136868e-13
190,3,223.70,671.10,671.10,1.136868e-13
202,3,253.98,761.94,761.94,1.136868e-13
225,3,273.96,821.88,821.88,1.136868e-13
254,3,263.19,789.57,789.57,1.136868e-13


# Floating-Point Precision Handling

Small numerical differences were detected during financial validation due to floating-point precision limitations in Python.

To perform accurate financial comparisons, `numpy.isclose()` was used with tolerance-based validation instead of direct equality checks.

This ensures that mathematically equivalent values are treated as valid.

In [13]:
invalid_total = df[
    ~np.isclose(
        df["CalculatedPrice"],
        df["TotalPrice"]
    )
]

invalid_total.shape

(0, 15)

# Text Standardization

Text-based columns were standardized to improve consistency across the dataset.

The following cleaning operations were performed:
- Removed leading/trailing spaces
- Applied consistent capitalization formatting

This ensures accurate grouping and analysis during future reporting and visualization tasks.

In [14]:
text_columns = [
    "Product",
    "PaymentMethod",
    "OrderStatus",
    "CouponCode",
    "ReferralSource",
    "ShippingAddress"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

In [15]:
df["Product"] = df["Product"].str.title()

df["PaymentMethod"] = df["PaymentMethod"].str.title()

df["OrderStatus"] = df["OrderStatus"].str.title()

df["ReferralSource"] = df["ReferralSource"].str.title()

df["CouponCode"] = df["CouponCode"].str.upper()

In [16]:
df["PaymentMethod"].unique()
df["OrderStatus"].unique()
df["ReferralSource"].unique()


<StringArray>
['Instagram', 'Referral', 'Email', 'Facebook', 'Google']
Length: 5, dtype: str

In [17]:
df["PaymentMethod"].unique()
df["OrderStatus"].unique()


<StringArray>
['Shipped', 'Cancelled', 'Returned', 'Delivered', 'Pending']
Length: 5, dtype: str

In [18]:
df["PaymentMethod"].unique()

<StringArray>
['Debit Card', 'Online', 'Credit Card', 'Gift Card', 'Cash']
Length: 5, dtype: str

# Removing Temporary Validation Columns

Temporary columns created during the validation process were removed to maintain a clean and production-ready dataset structure.

In [19]:
df.drop(columns=["CalculatedPrice"], inplace=True)

In [20]:
df.shape

(1200, 14)

# Exporting the Cleaned Dataset

The cleaned and validated dataset was exported for future analysis, reporting, and visualization purposes.

In [21]:
df.to_csv("../data/cleaned_ecommerce_data.csv", index=False)

# Project Conclusion

The raw e-commerce dataset was successfully cleaned and prepared for analysis.

## Tasks Completed
- Handled missing values
- Verified duplicate records
- Standardized text formatting
- Validated financial calculations
- Corrected floating-point comparison issues
- Ensured data consistency and integrity

## Final Result
The dataset is now clean, structured, and ready for:
- Exploratory Data Analysis (EDA)
- Dashboard Creation
- Business Reporting
- Predictive Analytics

This project demonstrates practical data cleaning and validation techniques using Python and Pandas.

In [22]:
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
